**Cell: 1**

# 11 - Exercises for notebook 10 - break the pipeline on purpose

Companion to `10_claude_add_new_phone_column.ipynb`. Notebook 10 showed the QA pipeline **working**. This one shows **where it breaks**:

- ⚙️ each exercise **sabotages one part** of the pipeline on purpose,
- 🧪 measures what the remaining checks **do and do not catch**,
- 🧪 then **writes the check that would have caught it**.

## Purpose

- ⚠️ **Prove the pipeline is load-bearing, not decorative.** Notebook 10 went green. Green proves nothing unless you have seen the same pipeline go **red for the right reason**. Each exercise removes or weakens one safeguard and shows the exact moment the remaining ones fail.
- ⚠️ **Show the failure modes of AI-assisted QA** *(Exercise 1 - attacks step 4's plan)*. A model only tests **the risks it is told about**. Exercise 1 deletes one risk from the test plan and watches the generated suite lose its root-cause test while still going red on symptoms - so a **symptom-level fix passes a weaker suite**.
- 🧪 **Turn one-off tests into structural tests** *(Exercise 2 - attacks the catalog)*. Exercise 2 adds a second untagged column (`date_of_birth`) and shows none of the seven phone-specific tests notice. The test you write walks **every** column in the catalog - it would have caught QA-114 **before it existed**.
- ✅ **Separate "masked" from "safe"** *(Exercise 3 - attacks the mask)*. Exercise 3 keeps the tag right and the mask applied, but leaks the first two characters through the hash. **Equality tests stay green**; the test you write checks **re-identification**, not just inequality.
- 💰 **Replace guessed budgets with measured ones** *(Exercise 4 - measures step 8's budget)*. Exercise 4 re-runs the step 8 fix call three times with `--output-format json`, reads `num_turns` from the envelope, and sets `--max-turns` from **observed maximum x 2.5** instead of from a number someone typed.
- 🧪 **Move a gate from the notebook into CI and attack it** *(Exercise 5 - attacks the CI gate)*. Exercise 5 pushes a branch where the fix also deletes the one assertion that would catch it. Which check notices? If none does, the `git diff --quiet -- tests/` gate from step 8 goes into `qa.yml`.
- 🧪 **Practise reading evidence, not output.** Every exercise leaves files under `qa_evidence/` - diffs, pytest logs, JSON envelopes. The answer to "did it work" is always **a file, never a feeling**.

## How it is built

- 📄 **Every exercise is independent** - each builds its own copy under `/tmp/qa_ex/exN` from the real repo at `/Users/satyak/Ranagr_DataCatalog_Iceberg`, so you can run them in any order, or re-run one without disturbing the others.
- 🤖 **Real `claude -p` calls** where the exercise is fundamentally about a live model step (1, 4).
- 🧪 **Deterministic Python/bash** everywhere notebook 10 already showed those parts are pure file edits (2, 3, 5) - a human is the one asked to "write the test" there, so that's what this notebook does.


## Where each exercise hits notebook 10's pipeline

```mermaid
flowchart TB
    NB10["Notebook 10 pipeline"] --> PLAN["Test plan (step 4)"]
    NB10 --> CAT["Catalog tags (core/catalog.py)"]
    NB10 --> MASK["Ranger mask (core/ranger.py)"]
    NB10 --> BUD["Turn budget (step 8)"]
    NB10 --> CI["CI gate (qa.yml)"]
    E1["Ex 1: delete a risk"] -.->|attacks| PLAN
    E2["Ex 2: untagged date_of_birth"] -.->|attacks| CAT
    E3["Ex 3: leaky hash"] -.->|attacks| MASK
    E4["Ex 4: measure num_turns"] -.->|measures| BUD
    E5["Ex 5: delete the witness test"] -.->|attacks| CI
```


**Cell: 2**

## Shared setup

**What to run:** Cells 3-6 below, once, before any exercise. They only write scripts and fixtures under `/tmp/qa_ex/` - the exercises call them.

- 🌿 **`build_baseline.sh DEST`** - steps 0-2c from notebook 10: clean room, baseline commit, ticket, branch, then the QA-114 change **with the bug** notebook 10 walks through (`phone` added, **not** tagged in the catalog). The starting point exercises 1 and 4 need.
- 🧪 **`build_shipped.sh DEST`** - the above, **plus the real fix landed** (phone tagged PII, guardrails detects phone leaks) and the generated governance suite in place. The "PR already merged" starting point exercises 2, 3 and 5 need.


In [ ]:
%%bash
# Cell: 3
mkdir -p /tmp/qa_ex/fixtures
cat > /tmp/qa_ex/build_baseline.sh <<'SCRIPT'
#!/usr/bin/env bash
set -euo pipefail
DEST="$1"

rm -rf "$DEST"
mkdir -p "$DEST"
rsync -a --exclude '.git' --exclude '__pycache__' --exclude '.DS_Store' \
  /Users/satyak/Ranagr_DataCatalog_Iceberg/ "$DEST/"
mkdir -p "$DEST/qa_evidence" "$DEST/prompts" "$DEST/tickets"
printf '__pycache__/\n*.pyc\n' > "$DEST/.gitignore"

git -C "$DEST" init -b main -q
git -C "$DEST" config user.email "qa@ambaricloud.com"
git -C "$DEST" config user.name  "QA Lead"
git -C "$DEST" add -A
git -C "$DEST" commit -qm "chore: baseline of the app under test"

cat > "$DEST/tickets/QA-114.md" <<'EOF'
# QA-114 - Add customer phone number to the profile

## Acceptance criteria
1. phone is present in the seed data for every one of the 16 customers.
2. phone appears in the default selected columns of the planner.
3. dave (admin) sees the raw phone number.
4. alice (analyst) must NOT see a raw phone number.
5. carol (support) must NOT see a raw phone number, but must see enough of it to confirm identity.
6. Nothing about the existing email / ssn / card behaviour changes.
EOF

git -C "$DEST" switch -c feature/qa-114-phone -q

python3 - "$DEST" <<'PYEOF'
import re, sys
dest = sys.argv[1]
data_py = f"{dest}/core/data.py"
src = open(data_py).read()
block = '''_ROWS = [
    # customer_id, name, city, state, region, email, ssn, card, phone, plan, total_spend
    (1001, "Maria Lopez",  "Bolingbrook", "IL", "US", "maria.lopez@example.com",  "859-45-5012", "3657-2286-1679-8935", "630-555-0101", "Gold",     14250.75),
    (1002, "James Carter", "Bolingbrook", "IL", "US", "james.carter@example.com", "421-88-7741", "4111-1111-1111-1234", "630-555-0102", "Platinum", 22890.10),
    (1003, "Priya Nair",   "Bolingbrook", "IL", "US", "priya.nair@example.com",   "330-21-9087", "5500-0000-0000-0004", "630-555-0103", "Gold",     19980.00),
    (1004, "Wei Zhang",    "Bolingbrook", "IL", "US", "wei.zhang@example.com",    "612-77-3344", "6011-0009-9013-9424", "630-555-0104", "Silver",    8420.40),
    (1005, "Aisha Khan",   "Bolingbrook", "IL", "US", "aisha.khan@example.com",   "788-12-6655", "3782-8224-6310-005",  "630-555-0105", "Platinum", 31200.55),
    (1006, "David Cohen",  "Bolingbrook", "IL", "US", "david.cohen@example.com",  "205-66-1290", "4012-8888-8888-1881", "630-555-0106", "Gold",     16730.25),
    (1007, "Sofia Rossi",  "Bolingbrook", "IL", "US", "sofia.rossi@example.com",  "551-09-4432", "4222-2222-2222-2225", "630-555-0107", "Silver",    9510.00),
    (1008, "Liam Murphy",  "Bolingbrook", "IL", "US", "liam.murphy@example.com",  "147-33-8821", "5105-1051-0510-5100", "630-555-0108", "Gold",     12740.90),
    (1009, "Hana Sato",    "Bolingbrook", "IL", "US", "hana.sato@example.com",    "962-40-1177", "3530-1113-3330-0000", "630-555-0109", "Platinum", 27640.30),
    (1010, "Omar Haddad",  "Bolingbrook", "IL", "US", "omar.haddad@example.com",  "418-55-7203", "6304-0000-0000-0000", "630-555-0110", "Gold",     15310.65),
    (1011, "Grace Lee",    "Bolingbrook", "IL", "US", "grace.lee@example.com",    "377-19-5560", "4917-4842-3535-0001", "630-555-0111", "Silver",    7020.10),
    (1012, "Noah Becker",  "Naperville",  "IL", "US", "noah.becker@example.com",  "880-23-4419", "4001-9192-5753-7193", "331-555-0112", "Gold",     11890.00),
    (1013, "Emma Wilson",  "Aurora",      "IL", "US", "emma.wilson@example.com",  "263-71-9902", "5200-8282-8282-8210", "331-555-0113", "Platinum", 20410.85),
    (1014, "Lucas Meyer",  "Munich",      "BY", "EU", "lucas.meyer@example.de",   "551-02-3380", "4000-0566-5566-5556", "+49-89-555-0114", "Gold",     17850.20),
    (1015, "Chloe Dubois", "Paris",       "75", "EU", "chloe.dubois@example.fr",  "204-88-1142", "5555-5555-5555-4444", "+33-1-555-0115", "Silver",    9930.45),
    (1016, "Mateo Garcia", "Madrid",      "28", "EU", "mateo.garcia@example.es",  "709-31-6650", "4242-4242-4242-4242", "+34-91-555-0116", "Platinum", 24560.00),
]

_COLS = ["customer_id", "name", "city", "state", "region",
         "email", "ssn", "card", "phone", "plan", "total_spend"]
'''
src = re.sub(r'_ROWS = \[.*?_COLS = \[.*?\]\n', block, src, flags=re.S)
open(data_py, "w").write(src)
PYEOF

sed -i '' 's|"card", "total_spend"\]|"card", "phone", "total_spend"]|' "$DEST/core/planner.py"
sed -i '' 's|        Column("plan",        "string"),|        Column("plan",        "string"),\n        Column("phone",       "string"),|' "$DEST/core/catalog.py"

git -C "$DEST" add -A
git -C "$DEST" commit -qm "feat(QA-114): add phone column to customer profile"
echo "baseline ready: $DEST"
SCRIPT
chmod +x /tmp/qa_ex/build_baseline.sh
echo "build_baseline.sh written"


In [ ]:
%%bash
# Cell: 4
cat > /tmp/qa_ex/fixtures/test_governance.py <<'FIXEOF'
import os, sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import pandas as pd
from core import login, ranger, guardrails, mcp_server
from core.data import load_table
from core.planner import plan_sql
from core import catalog

TABLE = "polaris.market.dimcustomer"

_RAW_US_PHONES = {
    "630-555-0101", "630-555-0102", "630-555-0103", "630-555-0104",
    "630-555-0105", "630-555-0106", "630-555-0107", "630-555-0108",
    "630-555-0109", "630-555-0110", "630-555-0111", "331-555-0112",
    "331-555-0113",
}
_RAW_EU_PHONES = {"+49-89-555-0114", "+33-1-555-0115", "+34-91-555-0116"}
_RAW_PHONES = _RAW_US_PHONES | _RAW_EU_PHONES


def test_phone_is_classified_in_catalog():
    meta = catalog.describe(TABLE)
    assert "phone" in meta["classified_columns"]
    phone_col = next(c for c in meta["columns"] if c["name"] == "phone")
    assert phone_col["tag"] == "PII"


def test_admin_sees_raw_phone():
    dave = login("dave")
    df, decision = ranger.enforce(dave, TABLE, load_table(TABLE))
    assert decision.masked_columns == {}
    for val in df["phone"]:
        assert val in _RAW_PHONES


def test_analyst_phone_is_masked():
    alice = login("alice")
    df, decision = ranger.enforce(alice, TABLE, load_table(TABLE))
    assert "phone" in decision.masked_columns
    for val in df["phone"]:
        assert val not in _RAW_PHONES


def test_support_phone_is_masked():
    carol = login("carol")
    df, decision = ranger.enforce(carol, TABLE, load_table(TABLE))
    assert len(df) > 0
    assert "phone" in decision.masked_columns
    for val in df["phone"]:
        assert val not in _RAW_EU_PHONES
    raw_eu = "+49-89-555-0114"
    last4 = "".join(ch for ch in raw_eu if ch.isdigit())[-4:]
    assert any(last4 in str(v) for v in df["phone"])


def test_guardrails_detects_raw_phone():
    alice = login("alice")
    df = pd.DataFrame({"phone": ["630-555-0101"]})
    decision = ranger.RangerDecision(
        role="analyst", tenant="US",
        masked_columns={"phone": "SHA-256 hash (PII)"},
    )
    audit = guardrails.scan_and_audit(alice, TABLE, df, decision)
    assert "PHONE" in audit.leaks


def test_guardrails_does_not_false_alarm():
    alice = login("alice")
    df = pd.DataFrame({
        "customer_id": [1001],
        "plan": ["Gold"],
        "total_spend": [14250.75],
        "city": ["Bolingbrook"],
    })
    decision = ranger.RangerDecision(
        role="analyst", tenant="US",
        masked_columns={"phone": "SHA-256 hash (PII)"},
    )
    audit = guardrails.scan_and_audit(alice, TABLE, df, decision)
    assert audit.leaks == []


def test_no_leaks_for_non_admin_query():
    alice = login("alice")
    plan = plan_sql("Top 10 Customers who live in Bollingbrook")
    _df, _decision, audit = mcp_server.run_query(alice, plan)
    assert audit.leaks == []


def test_existing_masking_unchanged():
    alice = login("alice")
    df, decision = ranger.enforce(alice, TABLE, load_table(TABLE))
    assert (df["card"] == "0000-0000-0000-0000").all()
    assert (df["ssn"] != "859-45-5012").all()
    assert "email" in decision.masked_columns
    assert "ssn" in decision.masked_columns
    assert "card" in decision.masked_columns
FIXEOF
echo "fixtures/test_governance.py written ($(wc -l < /tmp/qa_ex/fixtures/test_governance.py) lines)"


In [ ]:
%%bash
# Cell: 5
cat > /tmp/qa_ex/fixtures/guardrails.py <<'FIXEOF'
"""Guardrails - defense in depth, on top of Ranger.

Last line of defense: scan the (already enforced) output for raw PII/PCI that
should never appear if masking worked, then write a structured audit record.
For admins, raw values are expected, so leak alarms are suppressed.
"""
from __future__ import annotations
from dataclasses import dataclass, field
import re
import pandas as pd

_SSN = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")
_CARD = re.compile(r"\b(?:\d[ -]?){13,16}\b")
_EMAIL = re.compile(r"\b[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}\b")
# US NXX-NXX-XXXX or international +CC-area-exchange-subscriber
_PHONE = re.compile(r"\b\d{3}-\d{3}-\d{4}\b|\+\d{1,3}-\d{1,4}-\d{3}-\d{4}\b")


@dataclass
class AuditRecord:
    user: str
    role: str
    tenant: str
    table: str
    rows: int
    classified_columns: list = field(default_factory=list)
    leaks: list = field(default_factory=list)
    masked: dict = field(default_factory=dict)
    row_filter: str = ""

    def as_line(self) -> str:
        return (f"user={self.user} role={self.role} tenant={self.tenant} "
                f"table={self.table} rows={self.rows} "
                f"masked={list(self.masked)} filter='{self.row_filter}' "
                f"leaks={len(self.leaks)}")


def _is_masked_token(s: str) -> bool:
    """True if a regex hit is actually a recognized mask output, not a real leak."""
    digits = "".join(ch for ch in s if ch.isdigit())
    if digits and set(digits) <= {"0"}:        # 0000-0000-0000-0000 redaction
        return True
    if s.startswith("XXXXX") or "\u2026" in s or "***" in s:
        return True
    return False


def scan_and_audit(principal, table_fqn, df: pd.DataFrame, decision) -> AuditRecord:
    leaks = []
    if not principal.is_admin and len(df):
        blob = "\n".join(df.astype(str).apply(lambda r: " ".join(r), axis=1))
        for label, rx in (("SSN", _SSN), ("CARD", _CARD), ("EMAIL", _EMAIL), ("PHONE", _PHONE)):
            hits = [m for m in rx.findall(blob) if not _is_masked_token(str(m))]
            if not hits:
                continue
            if label == "EMAIL" and "email" not in decision.masked_columns:
                continue
            if label == "PHONE" and "phone" not in decision.masked_columns:
                continue
            leaks.append(label)
    return AuditRecord(
        user=principal.username, role=principal.role, tenant=principal.tenant,
        table=table_fqn, rows=len(df),
        classified_columns=list(decision.masked_columns),
        leaks=leaks, masked=decision.masked_columns, row_filter=decision.row_filter,
    )
FIXEOF
echo "fixtures/guardrails.py written ($(wc -l < /tmp/qa_ex/fixtures/guardrails.py) lines)"


In [ ]:
%%bash
# Cell: 6
cat > /tmp/qa_ex/build_shipped.sh <<'SCRIPT'
#!/usr/bin/env bash
set -euo pipefail
DEST="$1"

bash /tmp/qa_ex/build_baseline.sh "$DEST"

sed -i '' 's|Column("phone",       "string"),|Column("phone",       "string", tag="PII"),|' "$DEST/core/catalog.py"
cp /tmp/qa_ex/fixtures/guardrails.py "$DEST/core/guardrails.py"
mkdir -p "$DEST/tests"
cp /tmp/qa_ex/fixtures/test_governance.py "$DEST/tests/test_governance.py"

git -C "$DEST" add -A
git -C "$DEST" commit -qm "fix(QA-114): tag phone PII, detect phone leaks, add governance suite"
echo "shipped baseline ready: $DEST"
SCRIPT
chmod +x /tmp/qa_ex/build_shipped.sh
bash /tmp/qa_ex/build_shipped.sh /tmp/qa_ex/_shipped_check
python3 -m pytest -q /tmp/qa_ex/_shipped_check/tests/test_governance.py
python3 /tmp/qa_ex/_shipped_check/tests/test_flow.py > /dev/null 2>&1
echo "sanity check: shipped baseline is green on both suites"


**Cell: 7**

---
## Exercise 1 - Break the plan, not the code

> Delete the `phone` risk from `04_test_plan.json` and re-run step 5. The generated suite loses
> its root-cause test - and step 6 still goes red, on symptoms only. Then run step 8 and watch a
> symptom-level fix pass a weaker suite. *This is the failure mode of AI-assisted QA, reproduced
> on purpose.*

*This attacks **step 4's test plan** - the input that tells the model what to test.*

- 🧪 **The setup** - delete one risk (the untagged-`phone` root cause) from the test plan, then regenerate the suite from the weakened plan.
- ⚠️ **What to run** - Cells 8-14 below: build the room and write the plan (Cell 8), strip the risk (Cell 9), regenerate the suite with a real `claude -p` call (Cell 10), inspect what the suite lost (Cell 11), go RED (Cell 12), let a `claude -p` developer "fix" it (Cell 13), and see what green now means (Cell 14).
- ⚠️ **What you should observe** - the regenerated suite has **no root-cause test**, still goes **red on symptoms**, and the fix turns it green **without ever tagging `phone` in the catalog**.
- 💡 **The lesson**
  - 🧪 A model only tests **the risks it is told about** - delete a risk from the plan and its test silently disappears downstream.
  - ⚠️ Red is not enough: a suite that is red **for the wrong reason** can be satisfied by a **symptom-level fix**.
- 🧪 **The fix / the test that catches it** - hardcode the root-cause test in the step 5 prompt (what notebook 10's `writetests.md` does) instead of trusting the plan alone.

```mermaid
flowchart LR
    A["plan minus phone risk"] --> B["suite: no root-cause test"] --> C["RED - symptoms only"] --> D["symptom-level fix"] --> E["GREEN - catalog still wrong"]
```

**Why this needs a plan-driven prompt first:**

- 🧪 The catch: notebook 10's `writetests.md` **hardcodes** the eight tests by name - deleting a risk from the plan wouldn't change its output at all, since it never reads the plan's risk list to decide what to write.
- 🔹 That hardcoding *is* the safety net notebook 10 relies on.
- 🧪 So step 5 here uses a **plan-driven** prompt instead: one test per risk, named from the risk's own `test_name`. Now the plan is **load-bearing**, and deleting a risk from it has a real effect downstream.


In [ ]:
%%bash
# Cell: 8
D=/tmp/qa_ex/ex1
bash /tmp/qa_ex/build_baseline.sh "$D"

cat > "$D/prompts/testplan.md" <<EOF
You are a QA Lead on a data governance platform. You are writing the TEST PLAN for a change,
before any test code exists. You may read the repository but you may not change it.

The application: an agent turns English into SQL; Apache Ranger (simulated in core/ranger.py)
enforces a row-level tenant filter and column masking. The masks Ranger applies are derived from
classification tags declared in core/catalog.py - a column with no tag is, by definition, not
masked for anyone. core/guardrails.py is a second, independent scan of the output that looks for
raw sensitive values that should never have survived masking.

Read these files before answering:
  $D/tickets/QA-114.md      - what was requested
  $D/core/catalog.py        - the classification spec
  $D/core/ranger.py         - the enforcement point
  $D/core/guardrails.py     - the output scanner
  $D/tests/test_flow.py     - the suite that exists today

Produce the risk surface of this change. For each risk say what could be wrong, how severe it is,
and the name of the test that would prove it. Cover at least: the three roles (admin dave,
analyst alice, support carol), the classification metadata itself, the output scanner, and
regression of the existing email/ssn/card behaviour.

Output ONLY a JSON object, no prose, no code fences:
{
  "change_summary": "one sentence",
  "risks": [
    {"id":"R1","risk":"...","severity":"critical|high|medium|low","test_name":"test_...","why_it_matters":"..."}
  ],
  "regression_areas": ["..."],
  "out_of_scope": ["..."]
}

The diff under test:

EOF

git -C "$D" diff main...HEAD > "$D/qa_evidence/04_change.diff"

claude -p "$(cat "$D/prompts/testplan.md")$(cat "$D/qa_evidence/04_change.diff")" \
  --model claude-sonnet-4-6 \
  --allowedTools "Read,Grep,Glob" \
  --add-dir "$D" \
  --max-turns 12 </dev/null > "$D/qa_evidence/04_test_plan.json"

python3 -c "
import pathlib, sys
p = pathlib.Path('$D/qa_evidence/04_test_plan.json'); s = p.read_text()
if '{' not in s or '}' not in s:
    sys.exit(f'NO JSON: {s[:200]}')
p.write_text(s[s.index('{'):s.rindex('}') + 1])"

python3 -m json.tool "$D/qa_evidence/04_test_plan.json"


In [ ]:
%%bash
# Cell: 9
D=/tmp/qa_ex/ex1
python3 -c "
import json
p = '$D/qa_evidence/04_test_plan.json'
plan = json.load(open(p))
before = len(plan['risks'])
def is_root_cause(r):
    blob = (r.get('test_name', '') + ' ' + r.get('risk', '') + ' ' + r.get('why_it_matters', '')).lower()
    return any(kw in blob for kw in ('classif', 'catalog', ' tag', '\"tag\"'))
plan['risks'] = [r for r in plan['risks'] if not is_root_cause(r)]
after = len(plan['risks'])
json.dump(plan, open(p, 'w'), indent=2)
print(f'risks before: {before}  after: {after}  (only the catalog/classification risk removed - other phone risks kept)')
for r in plan['risks']:
    print(' -', r['id'], r['test_name'])
"


In [ ]:
%%bash
# Cell: 10
D=/tmp/qa_ex/ex1
cat > "$D/prompts/writetests_plandriven.md" <<'EOF'
You are a QA Lead writing an executable test suite. Write exactly one new file:

    tests/test_governance.py

Write ONE pytest test per risk in the JSON test plan below, named after that risk's "test_name".
Do not add tests for risks that are not listed. Do not invent a test for catalog classification
unless a risk in the plan explicitly asks for one.

Plain pytest. Module-level `def test_...` functions, plain `assert`, no classes, no fixtures.
Start with this bootstrap so it runs from any working directory:
    import os, sys
    sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
Import only from the app's public surface:
    from core import login, ranger, guardrails, mcp_server
    from core.data import load_table
    from core.planner import plan_sql
    from core import catalog
TABLE = "polaris.market.dimcustomer"

Write the file. Do not modify any other file. Do not run pytest. Reply with one line when done.

The test plan:

EOF

claude -p "$(cat "$D/prompts/writetests_plandriven.md")$(cat "$D/qa_evidence/04_test_plan.json")" \
  --model claude-sonnet-4-6 \
  --allowedTools "Read,Write" \
  --permission-mode acceptEdits \
  --add-dir "$D" \
  --max-turns 15 </dev/null > "$D/qa_evidence/05_write_reply.txt"

cat "$D/qa_evidence/05_write_reply.txt"
echo "---"
grep -o "^def test_[a-z_]*" "$D/tests/test_governance.py"


In [ ]:
%%bash
# Cell: 11
D=/tmp/qa_ex/ex1
echo "root-cause test present?"
if grep -q "def test_.*classif\|def test_phone_is_classified" "$D/tests/test_governance.py"; then
  echo "YES - unexpected, the exercise did not reproduce"
else
  echo "NO - confirmed: the generated suite has no test on core/catalog.py's phone tag"
fi


In [ ]:
%%bash
# Cell: 12
D=/tmp/qa_ex/ex1
python3 -m pytest -q "$D/tests/test_governance.py" > "$D/qa_evidence/06_red.txt" 2>&1 || true
cat "$D/qa_evidence/06_red.txt"
echo "---"
grep -c "failed" "$D/qa_evidence/06_red.txt"


In [ ]:
%%bash
# Cell: 13
D=/tmp/qa_ex/ex1
cat > "$D/prompts/fix.md" <<'EOF'
You are a developer. Make every test in tests/test_governance.py pass, and keep tests/test_flow.py
passing. Do NOT modify anything under tests/. Read core/catalog.py, core/ranger.py and
core/guardrails.py first. When done, reply with a two-line summary of the files you changed and
why. Do not run pytest.

The failing suite:

EOF
python3 -m pytest -q "$D/tests/test_governance.py" > /tmp/ex1_fail.txt 2>&1 || true

claude -p "$(cat "$D/prompts/fix.md")$(cat /tmp/ex1_fail.txt)" \
  --model claude-sonnet-4-6 \
  --allowedTools "Read,Edit" \
  --permission-mode acceptEdits \
  --add-dir "$D" \
  --max-turns 15 </dev/null > "$D/qa_evidence/08_fix_reply.txt"

cat "$D/qa_evidence/08_fix_reply.txt"


In [ ]:
%%bash
# Cell: 14
D=/tmp/qa_ex/ex1
echo "=== weaker suite, after the fix ==="
python3 -m pytest -q "$D/tests/test_governance.py"

echo
echo "=== but is the root cause actually fixed? ==="
if grep -q 'Column("phone", *"string", *tag="PII")' "$D/core/catalog.py"; then
  echo "catalog.py: phone IS tagged PII - the fix happened to be correct anyway"
else
  echo "catalog.py: phone is STILL untagged - the weaker suite went green on a symptom-level fix"
  grep -n 'Column("phone"' "$D/core/catalog.py"
fi
git -C "$D" diff -- core/ | head -60


**Cell: 15**

**Reading the result:**

- 🧪 With the root-cause risk stripped from the plan, step 5 never writes a test that opens `core/catalog.py` at all - every remaining test only observes *behaviour* through `ranger.enforce()`.
- 🧪 A developer under `--allowedTools "Read,Edit"` can satisfy that suite by **patching symptoms** inside `ranger.py` or `guardrails.py` (e.g. special-casing the column name) without ever adding `tag="PII"` to the catalog.
- ⚠️ The suite goes green; **the spec - the catalog - is still wrong**.
- ⚠️ This is exactly the failure mode notebook 10's step 5 prompt is designed to prevent by **hardcoding the root-cause test** instead of trusting the plan alone.


**Cell: 16**

---
## Exercise 2 - Add a second untagged column

> Give the customer a `date_of_birth` in `core/data.py` and `core/catalog.py`, tagged nothing.
> Which of the seven tests catches it? Write the one that should: a test that walks *every* column
> in the catalog and asserts each is either classified or on an explicit allow-list of public
> columns. That test would have caught QA-114 before it was written.

*This attacks **the catalog** - the same stage QA-114 itself broke, one column over.*

- 📄 **The setup** - add `date_of_birth` to `core/data.py` and `core/catalog.py` with **no tag**, on the shipped repo where the phone fix already landed.
- ✅ **What to run** - Cells 17-22 below: build the shipped repo (Cell 17), confirm both suites green (Cell 18), mutate (Cell 19), re-run and watch nothing notice (Cell 20), append the completeness test (Cell 21), tag the column and go green (Cell 22).
- ✅ **What you should observe** - all seven phone-specific tests **stay green** while raw dates of birth flow to every role.
- 💡 **The lesson**
  - 🧪 Tests written for one incident are **one-off tests**: they defend the last column, not the next one.
  - 🧪 The **structural** version - walk *every* column, assert each is classified or on an explicit public allow-list - catches the whole class, including QA-114 **before it was written**.
- 🧪 **The fix / the test that catches it** - `test_every_column_is_classified_or_public` (Cell 21); the fix is one `tag="PII"` in `core/catalog.py` (Cell 22).


In [ ]:
%%bash
# Cell: 17
bash /tmp/qa_ex/build_shipped.sh /tmp/qa_ex/ex2

In [ ]:
%%bash
# Cell: 18
D=/tmp/qa_ex/ex2
echo "=== before the mutation: both suites green ==="
python3 -m pytest -q "$D/tests/test_governance.py"
python3 "$D/tests/test_flow.py" > /dev/null 2>&1 && echo "test_flow.py: green"


In [ ]:
%%bash
# Cell: 19
D=/tmp/qa_ex/ex2
python3 - "$D/core/data.py" <<'PYEOF'
import re, sys
p = sys.argv[1]
src = open(p).read()

dobs = {
    1001: "14-Mar-1985", 1002: "02-Nov-1979", 1003: "22-Jul-1990", 1004: "09-Jan-1988",
    1005: "30-May-1993", 1006: "17-Sep-1975", 1007: "04-Dec-1991", 1008: "25-Jun-1982",
    1009: "11-Aug-1987", 1010: "19-Apr-1980", 1011: "27-Feb-1995", 1012: "06-Oct-1984",
    1013: "21-Mar-1978", 1014: "15-Jul-1989", 1015: "28-Nov-1992", 1016: "03-Jan-1986",
}
# non-numeric-dashed format on purpose: an all-digit "YYYY-MM-DD" can accidentally string
# together with an adjacent field across the blob's row-join and false-trip the CARD regex
# (13-16 digit run) - a real artifact of guardrails' blob-scanning approach, but a distraction
# from this exercise's actual point.

pattern = re.compile(
    r'\((?P<cid>\d+), (?P<mid>.*?),\s*(?P<phone>"[+\d][\d\-]+"),\s*'
    r'(?P<tail>"(?:Gold|Platinum|Silver)",\s*[\d.]+\),)'
)

def repl(m):
    cid = int(m.group("cid"))
    return f'({m.group("cid")}, {m.group("mid")}, {m.group("phone")}, "{dobs[cid]}", {m.group("tail")}'

new_src, n = pattern.subn(repl, src)
assert n == 16, f"expected 16 row substitutions, got {n}"
new_src = new_src.replace(
    '"email", "ssn", "card", "phone", "plan", "total_spend"]',
    '"email", "ssn", "card", "phone", "date_of_birth", "plan", "total_spend"]'
)
open(p, "w").write(new_src)
print("date_of_birth column inserted into all 16 rows")
PYEOF

sed -i '' 's|        Column("phone",       "string", tag="PII"),|        Column("phone",       "string", tag="PII"),\n        Column("date_of_birth", "string"),|' "$D/core/catalog.py"
sed -i '' 's|"card", "phone", "total_spend"\]|"card", "phone", "date_of_birth", "total_spend"]|' "$D/core/planner.py"

grep -n 'date_of_birth' "$D/core/catalog.py" "$D/core/planner.py"


In [ ]:
%%bash
# Cell: 20
D=/tmp/qa_ex/ex2
echo "=== after the mutation, unfixed: still green? ==="
python3 -m pytest -q "$D/tests/test_governance.py"
python3 "$D/tests/test_flow.py" > /dev/null 2>&1 && echo "test_flow.py: green"
echo
echo "confirmed - date_of_birth ships raw to every non-admin role, and nothing above noticed."
python3 -c "
import sys; sys.path.insert(0, '$D')
from core import login, ranger
from core.data import load_table
T = 'polaris.market.dimcustomer'
alice_out, dec = ranger.enforce(login('alice'), T, load_table(T))
print('alice masked_columns:', list(dec.masked_columns))
print('alice date_of_birth (should be masked, is not):', alice_out['date_of_birth'].iloc[0])
"


In [ ]:
%%bash
# Cell: 21
D=/tmp/qa_ex/ex2
cat >> "$D/tests/test_governance.py" <<'EOF'


# Exercise 2: a column with no tag is unmasked for everyone. Nothing above walks the full
# column list, so QA-114's own bug - phone shipped untagged - would have sailed through too.
PUBLIC_COLUMNS = {"customer_id", "name", "city", "state", "plan", "total_spend"}


def test_every_column_is_classified_or_public():
    meta = catalog.describe(TABLE)
    unaccounted = [
        c["name"] for c in meta["columns"]
        if c["tag"] is None and c["name"] not in PUBLIC_COLUMNS
    ]
    assert unaccounted == [], f"columns with no tag and no public allow-list entry: {unaccounted}"
EOF

echo "=== new completeness test added, running just that test ==="
python3 -m pytest -q "$D/tests/test_governance.py::test_every_column_is_classified_or_public" > "$D/qa_evidence/ex2_red.txt" 2>&1 || true
cat "$D/qa_evidence/ex2_red.txt"
grep -c "failed" "$D/qa_evidence/ex2_red.txt"


In [ ]:
%%bash
# Cell: 22
D=/tmp/qa_ex/ex2
sed -i '' 's|        Column("date_of_birth", "string"),|        Column("date_of_birth", "string", tag="PII"),|' "$D/core/catalog.py"

echo "=== fixed: date_of_birth tagged PII, full suite ==="
python3 -m pytest -q "$D/tests/test_governance.py"


**Cell: 23**

---
## Exercise 3 - Break the mask, not the tag

> Make `_hash` in `core/ranger.py` return the first two characters of the raw value followed by
> the hash. The tag is right, the column is masked, the suite goes green - and the data is
> re-identifiable. Write the test that fails.

*This attacks **the mask itself** (`core/ranger.py`) - the tag and the enforcement stay correct.*

- 🔹 **The setup** - mutate `_hash` so every "masked" value starts with the first two characters of the **raw** value.
- ✅ **What to run** - Cells 24-28 below: build and confirm green (Cell 24), mutate the hash (Cell 25), re-run the suite (Cell 26), append the re-identification test (Cell 27), revert the mutant and go green (Cell 28).
- ✅ **What you should observe** - every existing assertion **stays green**: the masked value is still `!=` the raw value, so equality tests are satisfied while the data is re-identifiable.
- 💡 **The lesson**
  - 🔹 "not equal to the raw value" is **not** the same as "not derived from the raw value".
  - 🛡️ A masking test must assert **re-identification is impossible** (no raw prefix survives the mask), not just inequality.
- ⚠️ **The fix / the test that catches it** - `test_hash_mask_does_not_leak_raw_prefix` (Cell 27): red on the mutant, green after the revert - and it stays in the suite.


In [ ]:
%%bash
# Cell: 24
bash /tmp/qa_ex/build_shipped.sh /tmp/qa_ex/ex3
D=/tmp/qa_ex/ex3
echo "=== before the mutation: green ==="
python3 -m pytest -q "$D/tests/test_governance.py"


In [ ]:
%%bash
# Cell: 25
D=/tmp/qa_ex/ex3
python3 - "$D/core/ranger.py" <<'PYEOF'
import sys
p = sys.argv[1]
s = open(p).read()
old = 'def _hash(v: str) -> str:\n    return hashlib.sha256(str(v).encode()).hexdigest()[:16].upper() + "…"\n'
new = ('def _hash(v: str) -> str:\n'
       '    # mutant: leaks the first two raw characters ahead of the hash\n'
       '    return str(v)[:2] + hashlib.sha256(str(v).encode()).hexdigest()[:16].upper() + "…"\n')
assert old in s, "could not find _hash to mutate"
open(p, "w").write(s.replace(old, new))
print("_hash mutated: now leaks a 2-char raw prefix")
PYEOF


In [ ]:
%%bash
# Cell: 26
D=/tmp/qa_ex/ex3
echo "=== after the mutation, unfixed: still green? ==="
python3 -m pytest -q "$D/tests/test_governance.py"
echo
python3 -c "
import sys; sys.path.insert(0, '$D')
from core import login, ranger
from core.data import load_table
T = 'polaris.market.dimcustomer'
raw = load_table(T)
out, dec = ranger.enforce(login('alice'), T, raw)
print('raw phone[0]   :', raw['phone'].iloc[0])
print('masked phone[0]:', out['phone'].iloc[0], '  <- starts with the raw prefix')
"


In [ ]:
%%bash
# Cell: 27
D=/tmp/qa_ex/ex3
cat >> "$D/tests/test_governance.py" <<'EOF'


# Exercise 3: "not equal to the raw value" is not the same as "not derived from the raw value".
# A mask that leaks a raw prefix passes every existing assertion here - none of them check for
# partial leakage, only for exact equality against the full raw string.
def test_hash_mask_does_not_leak_raw_prefix():
    alice = login("alice")
    raw = load_table(TABLE)
    out, decision = ranger.enforce(alice, TABLE, raw)
    assert "phone" in decision.masked_columns
    for raw_val, masked_val in zip(raw["phone"], out["phone"]):
        prefix = str(raw_val)[:3]
        assert not str(masked_val).startswith(prefix), (
            f"masked value {masked_val!r} leaks raw prefix {prefix!r} from {raw_val!r}"
        )
EOF

echo "=== new leak-detection test added ==="
python3 -m pytest -q "$D/tests/test_governance.py::test_hash_mask_does_not_leak_raw_prefix" > "$D/qa_evidence/ex3_red.txt" 2>&1 || true
cat "$D/qa_evidence/ex3_red.txt"
grep -c "failed" "$D/qa_evidence/ex3_red.txt"


In [ ]:
%%bash
# Cell: 28
D=/tmp/qa_ex/ex3
git -C "$D" checkout -q -- core/ranger.py
echo "=== mutant reverted, full suite ==="
python3 -m pytest -q "$D/tests/test_governance.py"


**Cell: 29**

---
## Exercise 4 - Measure your own turn budgets

> Re-run the step 8 call with `--max-turns 40 --output-format json`, three times, and read
> `num_turns` from the envelope. Set the cap to your observed maximum times 2.5 and compare it to
> the 15 used here.

*This measures **step 8's `--max-turns` budget** - the one number in the pipeline that was guessed, not measured.*

- ⚠️ **The setup** - the defect-state repo, `tests/test_governance.py` present and RED, `core/` unfixed; `core/` is reset with `git checkout` between runs so all three runs start from the same point.
- 🤖 **What to run** - Cells 30-32 below: build the room (Cell 30), three real `claude -p` fix runs under a deliberately loose `--max-turns 40` cap (Cell 31), then read the three envelopes and compute the budget (Cell 32).
- ▶️ **What you should observe** - `num_turns` varies run to run; the recommended cap is **observed maximum x 2.5**, compared against the 15 notebook 10 used.
- 💡 **The lesson**
  - 💰 A budget protects you only if it comes from **measurement**, not from a number someone typed.
  - 🤖 `--output-format json` makes every agent call measurable - the envelope's `num_turns` is the instrument.


In [ ]:
%%bash
# Cell: 30
D=/tmp/qa_ex/ex4
bash /tmp/qa_ex/build_baseline.sh "$D"
mkdir -p "$D/tests"
cp /tmp/qa_ex/fixtures/test_governance.py "$D/tests/test_governance.py"
git -C "$D" add -A
git -C "$D" commit -qm "test: governance suite for QA-114 (unfixed core/, expected RED)"

python3 -m pytest -q "$D/tests/test_governance.py" > /dev/null 2>&1 || echo "confirmed RED before any fix"

cat > "$D/prompts/fix.md" <<'EOF'
You are a developer. Make every test in tests/test_governance.py pass, and keep tests/test_flow.py
passing. Do NOT modify anything under tests/. Read core/catalog.py, core/ranger.py and
core/guardrails.py first. When done, reply with a two-line summary of the files you changed and
why. Do not run pytest.

The failing suite:

EOF
python3 -m pytest -q "$D/tests/test_governance.py" > /tmp/ex4_fail.txt 2>&1 || true
echo "ex4 ready"


In [ ]:
%%bash
# Cell: 31
D=/tmp/qa_ex/ex4
rm -f /tmp/ex4_run1.json /tmp/ex4_run2.json /tmp/ex4_run3.json

for i in 1 2 3; do
  git -C "$D" checkout -q -- core/
  claude -p "$(cat "$D/prompts/fix.md")$(cat /tmp/ex4_fail.txt)" \
    --model claude-sonnet-4-6 \
    --allowedTools "Read,Edit" \
    --permission-mode acceptEdits \
    --add-dir "$D" \
    --max-turns 40 \
    --output-format json </dev/null > "/tmp/ex4_run${i}.json"
  turns=$(python3 -c "import json; print(json.load(open('/tmp/ex4_run${i}.json'))['num_turns'])")
  echo "run $i: num_turns = $turns"
done


In [ ]:
%%bash
# Cell: 32
python3 -c "
import json
turns = []
for i in (1, 2, 3):
    turns.append(json.load(open(f'/tmp/ex4_run{i}.json'))['num_turns'])
observed_max = max(turns)
suggested = round(observed_max * 2.5)
print('observed turns:', turns)
print('observed max  :', observed_max)
print('suggested cap :', suggested, '(observed max x 2.5)')
print('notebook 10 used: 15')
print('verdict       :', 'notebook 10\\'s cap holds' if suggested <= 15 else 'notebook 10\\'s cap of 15 is UNDERSIZED for this job - use ' + str(suggested))
"
git -C /tmp/qa_ex/ex4 checkout -q -- core/
echo "core/ reset to unfixed state"


**Cell: 33**

---
## Exercise 5 - Move a gate into CI and try to sneak past it

> Push a branch where the fix also edits `tests/test_governance.py` to delete an assertion. Which
> job catches it? If none does, add the `git diff --quiet -- tests/` check to `qa.yml` - the same
> command from step 8.

*This attacks **the CI gate** - what happens when notebook 10's step 8 gate lives only in the notebook.*

- 🧪 **The setup** - one commit that both reintroduces the QA-114-D1 defect (silently drops `PHONE` from `core/guardrails.py`'s scan) and deletes `test_guardrails_detects_raw_phone` - the one test that exercises it.
- 🌿 **What to run** - Cells 34-36 below: build the shipped repo plus `qa.yml` (Cell 34), make the sneaky commit on a branch (Cell 35), run the same jobs `qa.yml` would run (Cell 36).
- ⚠️ **What you should observe** - both test suites pass clean; only the `tests_unmodified` job (`git diff --quiet -- tests/`) goes red.
- 💡 **The lesson**
  - 🧪 A test suite cannot defend itself - a commit can **delete its own witness**.
  - 🧪 Gate on "**were the tests touched**", instead of trying to review every possible way a test could be weakened.
- 🧪 **The fix / the test that catches it** - the `tests_unmodified` job in `qa.yml`: the same `git diff --quiet -- tests/` one-liner from step 8, promoted into CI.

**Notes on the simulation:**

- 🧪 No GitHub remote here, so "push a branch" and "which job catches it" are simulated locally: the same checks `qa.yml` would run, run in this shell, against a branch that both reintroduces a bug and quietly deletes the one test that would have caught it.
- 🧪 The mutation has to be chosen carefully for the exercise to mean anything: reusing step 10's SSN-untag mutant (tried first, below in spirit) gets caught by `tests/test_flow.py` regardless of what happens to `tests/test_governance.py`, because the legacy suite checks SSN masking directly - deleting one governance assertion wouldn't have let anything through.
- 🧪 `phone` has **no such independent witness**: `test_flow.py` predates QA-114 and has never heard of it. So the sneak drops `PHONE` from the guardrails scan and deletes its only test **in the same commit**.


In [ ]:
%%bash
# Cell: 34
D=/tmp/qa_ex/ex5
bash /tmp/qa_ex/build_shipped.sh "$D"

mkdir -p "$D/.github/workflows"
cat > "$D/.github/workflows/qa.yml" <<'EOF'
name: qa

on:
  pull_request:
  push:
    branches: [main]

jobs:
  regression:
    name: legacy suite (must stay green)
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -r requirements.txt pytest
      - run: python3 tests/test_flow.py

  governance:
    name: governance suite (QA-114)
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -r requirements.txt pytest
      - run: python3 -m pytest -q tests/test_governance.py

  tests_unmodified:
    name: tests/ must not be edited by a fix
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
        with: { fetch-depth: 0 }
      # Step 8's own gate, promoted into CI: a "fix" that edits the spec is not a fix.
      - run: git diff --quiet origin/main...HEAD -- tests/

  review:
    name: security review of the diff
    runs-on: ubuntu-latest
    if: github.event.pull_request.head.repo.full_name == github.repository
    steps:
      - uses: actions/checkout@v4
        with: { fetch-depth: 0 }
      - run: npm install -g @anthropic-ai/claude-code
      - name: the diff under review
        run: git diff origin/main...HEAD > /tmp/final.diff
      - name: claude -p, read-only
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
        run: |
          claude -p "$(cat prompts/review.md)$(cat /tmp/final.diff)" \
            --model claude-sonnet-4-6 \
            --allowedTools "Read,Grep" \
            --max-turns 12 </dev/null > /tmp/review.json
          cat /tmp/review.json
      - name: the gate - blockers must be zero
        run: |
          grep -c '"severity": *"blocker"' /tmp/review.json > /tmp/blockers.txt || true
          grep -qx "0" /tmp/blockers.txt
EOF
git -C "$D" add -A
git -C "$D" commit -qm "ci: run the regression, governance, tests-unmodified and review gates on every PR"
git -C "$D" branch main_ref
echo "qa.yml committed, main_ref bookmark set"


In [ ]:
%%bash
# Cell: 35
D=/tmp/qa_ex/ex5
git -C "$D" switch -c sneaky/qa-114-shortcut -q

# The "fix": guardrails silently stops scanning for PHONE at all - QA-114-D1, reintroduced.
python3 - "$D/core/guardrails.py" <<'PYEOF'
import sys
p = sys.argv[1]
s = open(p).read()
old = '("SSN", _SSN), ("CARD", _CARD), ("EMAIL", _EMAIL), ("PHONE", _PHONE)'
new = '("SSN", _SSN), ("CARD", _CARD), ("EMAIL", _EMAIL)'
assert old in s
open(p, "w").write(s.replace(old, new))
print("guardrails: PHONE scanning silently dropped")
PYEOF

echo "before touching tests/, this is what would fail:"
python3 -m pytest -q "$D/tests/test_governance.py::test_guardrails_detects_raw_phone" 2>&1 | tail -5

# ...then quietly delete the one test in tests/ that would have caught it.
python3 - "$D/tests/test_governance.py" <<'PYEOF'
import re, sys
p = sys.argv[1]
s = open(p).read()
pattern = re.compile(r'\n\ndef test_guardrails_detects_raw_phone\(\):.*?(?=\n\ndef )', re.S)
new_s, n = pattern.subn('', s)
assert n == 1, n
open(p, "w").write(new_s)
print("deleted: test_guardrails_detects_raw_phone")
PYEOF

git -C "$D" add -A
git -C "$D" commit -qm "fix: tune guardrails scanning (sneaks a test out of tests/)"
echo "sneaky branch ready"


In [ ]:
%%bash
# Cell: 36
D=/tmp/qa_ex/ex5
echo "=== job: regression (legacy suite - predates phone entirely) ==="
python3 "$D/tests/test_flow.py" > /tmp/ex5_regression.txt 2>&1
echo "exit=$?  ->  $(tail -1 /tmp/ex5_regression.txt)"

echo
echo "=== job: governance (post-sneak suite) ==="
python3 -m pytest -q "$D/tests/test_governance.py" > /tmp/ex5_governance.txt 2>&1
rc=$?
tail -5 /tmp/ex5_governance.txt
echo "exit=$rc  ($([ $rc -eq 0 ] && echo 'PASSED - the missing test means nobody noticed' || echo 'caught it'))"

echo
echo "=== job: tests_unmodified (git diff --quiet origin/main...HEAD -- tests/) ==="
if git -C "$D" diff --quiet main_ref...sneaky/qa-114-shortcut -- tests/; then
  echo "exit=0  tests/ unchanged (would NOT catch this branch)"
else
  echo "exit=1  tests/ WAS modified  -> this is the job that catches it"
fi


**Cell: 37**

**Verdict:**

- ✅ Both the legacy suite and the (now seven-test) governance suite **pass clean** - `test_flow.py` was written before phone existed and has nothing to say about it, and the governance suite's only phone-leak witness **left with the commit** that reintroduced the leak.
- 🧪 Only **`tests_unmodified`** - `git diff --quiet -- tests/`, the same one-liner from notebook 10's step 8 gate, now promoted into `qa.yml` - notices the branch touched `tests/` at all, and **fails the PR regardless of what the edit was**.
- 🧪 That's the point of gating on "**were the tests touched**" instead of trying to review every possible way a test could be weakened.
